# 🗄️ Mineração de Dados
## Módulo 1: Introdução à Mineração de Dados e Descoberta de Conhecimento (KDD)

---

## 🎯 Objetivos deste Notebook

Ao concluir este notebook, você será capaz de:

1. ✅ Carregar e explorar datasets reais com Pandas e Scikit-learn
2. ✅ Realizar Análise Exploratória de Dados (EDA) completa
3. ✅ Implementar o pipeline CRISP-DM na prática
4. ✅ Comparar múltiplos algoritmos de classificação
5. ✅ Visualizar árvores de decisão e fronteiras de decisão
6. ✅ Analisar complexidade e escalabilidade de algoritmos

---

## 📋 Sumário

1. [Setup e Importações](#setup)
2. [Explorando Dados Reais — Dataset Wine](#wine)
3. [Análise Exploratória de Dados (EDA)](#eda)
4. [Pipeline CRISP-DM na Prática](#crispdm)
5. [Comparação de Algoritmos de Mineração](#comparacao)
6. [Visualizando o Conhecimento Descoberto](#visualizacao)
7. [Análise de Complexidade e Escalabilidade](#complexidade)
8. [Exercícios Práticos](#exercicios)


In [ ]:
# Instalação das dependências necessárias
# Execute esta célula apenas uma vez (especialmente no Google Colab)
!pip install -q scikit-learn pandas numpy matplotlib seaborn scipy plotly kaleido
print('Instalação concluída!')

## 📦 1. Setup e Importações <a id='setup'></a>

Importamos todas as bibliotecas necessárias e configuramos o ambiente para reprodutibilidade.


In [ ]:
# ============================================================
# IMPORTAÇÕES PRINCIPAIS
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import scipy
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
import time

# Algoritmos de Machine Learning
from sklearn.datasets import load_wine, make_classification
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

# Configurações globais
warnings.filterwarnings('ignore')          # Suprimir avisos para clareza
np.random.seed(42)                          # Semente para reprodutibilidade

# Estilo dos gráficos Matplotlib
plt.style.use('seaborn-v0_8-whitegrid')
matplotlib.rcParams['figure.figsize'] = (12, 6)
matplotlib.rcParams['font.size'] = 11
matplotlib.rcParams['axes.titlesize'] = 13
matplotlib.rcParams['axes.labelsize'] = 11
sns.set_palette('husl')                     # Paleta de cores harmoniosa

# Imprimir versões das bibliotecas para documentação
print('=' * 55)
print('VERSÕES DAS BIBLIOTECAS CARREGADAS')
print('=' * 55)
import sklearn
print(f'NumPy:        {np.__version__}')
print(f'Pandas:       {pd.__version__}')
print(f'Matplotlib:   {matplotlib.__version__}')
print(f'Seaborn:      {sns.__version__}')
print(f'Scikit-learn: {sklearn.__version__}')
print(f'SciPy:        {scipy.__version__}')
print('=' * 55)
print('Ambiente configurado com sucesso!')

## 🗄️ 2. Explorando Dados Reais — Dataset Wine <a id='wine'></a>

O **Dataset Wine** contém 178 amostras de vinhos italianos de 3 cultivares diferentes,
descritos por 13 atributos físico-químicos. É um clássico para estudar classificação
e análise exploratória de dados.

**Referência:** Forina et al. (1986) — UCI Machine Learning Repository


In [ ]:
# ============================================================
# CARREGANDO E ESTRUTURANDO O DATASET WINE
# ============================================================

# Carregar o dataset do scikit-learn
wine_data = load_wine()

# Converter para DataFrame do Pandas para facilitar a análise
df = pd.DataFrame(
    data=wine_data.data,
    columns=wine_data.feature_names   # Usar os nomes originais das features
)
df['target'] = wine_data.target        # Adicionar coluna alvo (0, 1, 2)
df['cultivar'] = df['target'].map({   # Mapear para nomes descritivos
    0: 'Cultivar_A',
    1: 'Cultivar_B',
    2: 'Cultivar_C'
})

# Informações básicas do dataset
print('=' * 55)
print('INFORMAÇÕES GERAIS DO DATASET WINE')
print('=' * 55)
print(f'Dimensões: {df.shape[0]} linhas x {df.shape[1]} colunas')
print(f'Features:  {len(wine_data.feature_names)}')
print(f'Classes:   {list(wine_data.target_names)}')
print()

# Primeiras linhas
print('Primeiras 5 linhas do dataset:')
display(df.head())

# Tipos de dados
print('\nTipos de dados por coluna:')
print(df.dtypes)

In [ ]:
# ============================================================
# DIAGNÓSTICO DETALHADO DOS DADOS
# ============================================================

print('VALORES AUSENTES POR COLUNA')
print('-' * 40)
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'Nenhum valor ausente encontrado!')

print('\nDISTRIBUIÇÃO DAS CLASSES (Target)')
print('-' * 40)
class_dist = df['cultivar'].value_counts()
for cls, count in class_dist.items():
    pct = count / len(df) * 100
    print(f'  {cls}: {count} amostras ({pct:.1f}%)')

print('\nESTATÍSTICAS DESCRITIVAS GERAIS')
print('-' * 40)
display(df.describe().round(2))

print('\nESTATÍSTICAS POR CULTIVAR (média de cada feature)')
print('-' * 40)
# Agrupar por cultivar e calcular médias
display(df.groupby('cultivar')[wine_data.feature_names].mean().round(2))

## 📊 3. Análise Exploratória de Dados (EDA) <a id='eda'></a>

A EDA é fundamental para entender a estrutura dos dados antes de aplicar qualquer algoritmo.
Visualizamos distribuições, correlações e separabilidade das classes.


In [ ]:
# ============================================================
# VISUALIZAÇÕES EDA — PAINEL PRINCIPAL (2x2)
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Análise Exploratória — Dataset Wine', fontsize=16, fontweight='bold', y=1.01)

# --- Gráfico 1: Distribuição das classes (barras) ---
ax1 = axes[0, 0]
colors_bar = ['#2196F3', '#FF9800', '#4CAF50']
class_counts = df['cultivar'].value_counts()
bars = ax1.bar(class_counts.index, class_counts.values, color=colors_bar, edgecolor='white', linewidth=1.5)
# Adicionar rótulos nas barras
for bar, count in zip(bars, class_counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
             f'{count}\n({count/len(df)*100:.0f}%)', ha='center', va='bottom', fontsize=10)
ax1.set_title('1. Distribuição das Classes', fontweight='bold')
ax1.set_xlabel('Cultivar')
ax1.set_ylabel('Número de Amostras')

# --- Gráfico 2: Mapa de calor de correlações ---
ax2 = axes[0, 1]
# Selecionar apenas colunas numéricas
corr_matrix = df[wine_data.feature_names].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # Ocultar triângulo superior
sns.heatmap(corr_matrix, mask=mask, annot=False, cmap='coolwarm',
            center=0, vmin=-1, vmax=1, ax=ax2, cbar_kws={'shrink': 0.8})
ax2.set_title('2. Mapa de Correlações entre Features', fontweight='bold')
ax2.tick_params(axis='x', rotation=45, labelsize=7)
ax2.tick_params(axis='y', labelsize=7)

# --- Gráfico 3: Box plots das 4 features mais importantes ---
ax3 = axes[1, 0]
top_features = ['alcohol', 'total_phenols', 'flavanoids', 'color_intensity']
# Normalizar para comparação visual (z-score)
df_norm = df[top_features].copy()
for col in top_features:
    df_norm[col] = (df_norm[col] - df_norm[col].mean()) / df_norm[col].std()
ax3.boxplot([df_norm[f] for f in top_features], labels=top_features, patch_artist=True,
            boxprops=dict(facecolor='lightblue', color='navy'),
            medianprops=dict(color='red', linewidth=2))
ax3.set_title('3. Box Plots das Features Principais (Normalizadas)', fontweight='bold')
ax3.set_ylabel('Valor (z-score)')
ax3.tick_params(axis='x', rotation=15)

# --- Gráfico 4: Scatter plot das 2 primeiras features colorido por classe ---
ax4 = axes[1, 1]
colors_scatter = {0: '#2196F3', 1: '#FF9800', 2: '#4CAF50'}
for cultivar_id, color in colors_scatter.items():
    mask_cls = df['target'] == cultivar_id
    ax4.scatter(df[mask_cls]['alcohol'], df[mask_cls]['flavanoids'],
                c=color, label=f'Cultivar {["A","B","C"][cultivar_id]}',
                alpha=0.7, edgecolors='white', s=60)
ax4.set_title('4. Álcool vs. Flavanóides por Cultivar', fontweight='bold')
ax4.set_xlabel('Álcool (%)')
ax4.set_ylabel('Flavanóides')
ax4.legend(loc='best')

plt.tight_layout()
plt.show()
print('Painel EDA gerado com sucesso!')

In [ ]:
# ============================================================
# DISTRIBUIÇÃO DE TODAS AS 13 FEATURES
# ============================================================

features = wine_data.feature_names
n_features = len(features)      # 13 features
n_cols = 4
n_rows = (n_features + n_cols - 1) // n_cols  # Calcular linhas necessárias

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 14))
fig.suptitle('Distribuição de Todas as Features — Dataset Wine',
             fontsize=14, fontweight='bold', y=1.01)

# Paleta de cores para as 3 classes
palette = {'Cultivar_A': '#2196F3', 'Cultivar_B': '#FF9800', 'Cultivar_C': '#4CAF50'}

for idx, feature in enumerate(features):
    row, col = divmod(idx, n_cols)
    ax = axes[row, col]
    # Histograma com KDE para cada cultivar
    for cultivar, color in palette.items():
        data_subset = df[df['cultivar'] == cultivar][feature]
        sns.histplot(data_subset, kde=True, ax=ax, color=color,
                     alpha=0.4, label=cultivar, bins=15)
    ax.set_title(feature.replace('_', ' ').title(), fontsize=9, fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(labelsize=7)
    if idx == 0:  # Legenda apenas no primeiro gráfico
        ax.legend(fontsize=7)

# Ocultar eixos extras (caso n_features não preencha a grade)
for idx in range(n_features, n_rows * n_cols):
    row, col = divmod(idx, n_cols)
    axes[row, col].set_visible(False)

plt.tight_layout()
plt.savefig('wine_distributions.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Distribuições de {n_features} features plotadas!')
print('Figura salva como wine_distributions.png')

## ⚙️ 4. Pipeline CRISP-DM na Prática <a id='crispdm'></a>

Implementamos as fases do CRISP-DM de forma estruturada para o dataset Wine.
Cada fase é documentada e os resultados são apresentados de forma clara.


In [ ]:
# ============================================================
# IMPLEMENTAÇÃO DO PIPELINE CRISP-DM
# ============================================================

print('=' * 60)
print('PIPELINE CRISP-DM: CLASSIFICAÇÃO DE VINHOS')
print('=' * 60)

# --- FASE 1: Entendimento do Negócio ---
business_understanding = {
    'problema': 'Classificar automaticamente vinhos por cultivar',
    'objetivo_negocio': 'Automatizar controle de qualidade na produção',
    'metrica_sucesso': 'Acurácia >= 95% em dados de teste',
    'restricoes': 'Modelo deve ser interpretável pelo enólogo'
}
print('\n[FASE 1] Entendimento do Negócio')
for k, v in business_understanding.items():
    print(f'  {k.replace("_", " ").title()}: {v}')

# --- FASE 2: Entendimento dos Dados ---
print('\n[FASE 2] Entendimento dos Dados')
X = df[wine_data.feature_names].values   # Matriz de features
y = df['target'].values                   # Vetor alvo
print(f'  Amostras: {X.shape[0]}, Features: {X.shape[1]}, Classes: {len(np.unique(y))}')
print(f'  Valores ausentes: {np.isnan(X).sum()}')
print(f'  Range de valores: [{X.min():.2f}, {X.max():.2f}]')

# --- FASE 3: Preparação dos Dados ---
print('\n[FASE 3] Preparação dos Dados')
# Dividir em treino (80%) e teste (20%), estratificando por classe
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'  Treino: {X_train.shape[0]} amostras | Teste: {X_test.shape[0]} amostras')

# --- FASE 4: Modelagem com sklearn Pipeline ---
print('\n[FASE 4] Modelagem (Pipeline: Scaler + RandomForest)')
# Pipeline que garante que o scaler seja ajustado apenas nos dados de treino
pipeline = Pipeline([
    ('scaler', StandardScaler()),              # Passo 1: normalização
    ('clf', RandomForestClassifier(
        n_estimators=100,                      # 100 árvores no ensemble
        max_depth=5,                           # Profundidade máxima
        random_state=42
    ))
])
pipeline.fit(X_train, y_train)               # Treinar no conjunto de treino
y_pred = pipeline.predict(X_test)            # Predizer no conjunto de teste

# --- FASE 5: Avaliação ---
print('\n[FASE 5] Avaliação')
acc = accuracy_score(y_test, y_pred)
print(f'  Acurácia no Teste: {acc:.4f} ({acc*100:.2f}%)')
print()
print('  Relatório de Classificação:')
target_names = [f'Cultivar_{c}' for c in ['A', 'B', 'C']]
print(classification_report(y_test, y_pred, target_names=target_names, digits=4))

# --- FASE 6: Implantação (simulada) ---
print('[FASE 6] Implantação')
print('  Modelo treinado e pronto para deploy!')
print('  Próximo passo: serializar com joblib.dump() e servir via API REST.')

In [ ]:
# ============================================================
# VALIDAÇÃO CRUZADA (5-FOLD STRATIFIED CV)
# ============================================================

print('VALIDAÇÃO CRUZADA — 5-Fold Estratificado')
print('=' * 55)

# Configurar validação cruzada estratificada
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Executar CV e coletar scores por fold
scores = cross_val_score(
    pipeline, X, y,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1          # Usar todos os núcleos disponíveis
)

# Exibir resultados em tabela formatada
print(f"{'Fold':>6} | {'Acurácia':>10} | {'Desvio da Média':>15}")
print('-' * 38)
for i, score in enumerate(scores, 1):
    desvio = score - scores.mean()
    sinal = '+' if desvio >= 0 else ''
    print(f"{'Fold ' + str(i):>6} | {score:.4f}     | {sinal}{desvio:.4f}")
print('-' * 38)
print(f"{'Média':>6} | {scores.mean():.4f}     | ± {scores.std():.4f}")
print()
print(f'Acurácia média: {scores.mean()*100:.2f}% (±{scores.std()*100:.2f}%)')
print(f'Intervalo de confiança (95%): [{(scores.mean()-2*scores.std())*100:.2f}%, {(scores.mean()+2*scores.std())*100:.2f}%]')

## 🏆 5. Comparação de Algoritmos de Mineração <a id='comparacao'></a>

Comparamos 7 classificadores clássicos usando validação cruzada de 5 folds.
Avaliamos tanto a acurácia quanto o tempo de treinamento.


In [ ]:
# ============================================================
# COMPARAÇÃO DE 7 CLASSIFICADORES
# ============================================================

# Definir os 7 classificadores com seus nomes
classifiers = {
    'Logistic\nRegression':   LogisticRegression(max_iter=1000, random_state=42),
    'Decision\nTree':         DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random\nForest':         RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM':                    SVC(kernel='rbf', C=10, gamma='scale', random_state=42),
    'K-Nearest\nNeighbors':   KNeighborsClassifier(n_neighbors=5),
    'Naive\nBayes':           GaussianNB(),
    'Gradient\nBoosting':     GradientBoostingClassifier(n_estimators=100, random_state=42)
}

# Normalizar features para comparação justa
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Coletar resultados
results = {}
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('Avaliando classificadores...')
print('-' * 60)

for name, clf in classifiers.items():
    start = time.time()                          # Iniciar cronômetro
    scores_cv = cross_val_score(clf, X_scaled, y, cv=cv5, scoring='accuracy', n_jobs=-1)
    elapsed = time.time() - start                # Tempo decorrido
    results[name] = {
        'mean': scores_cv.mean(),
        'std':  scores_cv.std(),
        'time': elapsed
    }
    name_clean = name.replace('\n', ' ')
    print(f'{name_clean:30s} | Acc: {scores_cv.mean():.4f} ± {scores_cv.std():.4f} | Tempo: {elapsed:.2f}s')

# --- Gráfico de comparação com barras de erro ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Comparação de Classificadores — Dataset Wine', fontsize=14, fontweight='bold')

names = list(results.keys())
means = [results[n]['mean'] for n in names]
stds  = [results[n]['std']  for n in names]
times = [results[n]['time'] for n in names]

# Cores do melhor ao pior desempenho
colors_perf = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(names)))
sorted_idx = np.argsort(means)
colors_sorted = [colors_perf[i] for i in range(len(names))]

# Barras de acurácia
bars = ax1.barh(names, means, xerr=stds, color=colors_sorted,
                edgecolor='gray', linewidth=0.8, capsize=4, height=0.6)
ax1.set_xlabel('Acurácia (5-Fold CV)')
ax1.set_title('Acurácia Média com Barras de Erro')
ax1.set_xlim(0.7, 1.05)
for bar, mean in zip(bars, means):
    ax1.text(mean + 0.005, bar.get_y() + bar.get_height()/2.,
             f'{mean:.3f}', va='center', fontsize=9)

# Barras de tempo de treinamento
ax2.barh(names, times, color='steelblue', edgecolor='navy', linewidth=0.8, height=0.6)
ax2.set_xlabel('Tempo de Avaliação CV (segundos)')
ax2.set_title('Tempo de Treinamento + CV')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# GRÁFICO RADAR (SPIDER CHART) — 4 MÉTRICAS
# ============================================================

# Calcular 4 métricas por classificador
metric_names = ['Acurácia', 'Precisão', 'Recall', 'F1-Score']
metric_data = {}

for name, clf in classifiers.items():
    # Treinar e predizer com train/test split para métricas detalhadas
    clf.fit(X_scaled[:140], y[:140])            # Treino
    y_p = clf.predict(X_scaled[140:])           # Teste
    y_true = y[140:]
    metric_data[name] = [
        accuracy_score(y_true, y_p),
        precision_score(y_true, y_p, average='weighted', zero_division=0),
        recall_score(y_true, y_p, average='weighted', zero_division=0),
        f1_score(y_true, y_p, average='weighted', zero_division=0)
    ]

# Configurar o gráfico polar
n_metrics = len(metric_names)
angles = np.linspace(0, 2 * np.pi, n_metrics, endpoint=False).tolist()
angles += angles[:1]  # Fechar o polígono

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))

colors_radar = plt.cm.tab10(np.linspace(0, 1, len(classifiers)))

for (name, metrics), color in zip(metric_data.items(), colors_radar):
    values = metrics + metrics[:1]              # Fechar o polígono
    ax.plot(angles, values, 'o-', linewidth=2,
            label=name.replace('\n', ' '), color=color)
    ax.fill(angles, values, alpha=0.08, color=color)

# Eixos e rótulos
ax.set_xticks(angles[:-1])
ax.set_xticklabels(metric_names, fontsize=12, fontweight='bold')
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=8)
ax.set_title('Comparação de Classificadores — Radar de Métricas',
             size=13, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print('Gráfico radar gerado com sucesso!')

## 🔍 6. Visualizando o Conhecimento Descoberto <a id='visualizacao'></a>

Visualizamos a árvore de decisão treinada e as importâncias de features,
que representam o **conhecimento descoberto** pela mineração de dados.


In [ ]:
# ============================================================
# ÁRVORE DE DECISÃO — VISUALIZAÇÃO DO CONHECIMENTO
# ============================================================

fig, (ax_tree, ax_imp) = plt.subplots(1, 2, figsize=(20, 8))

# Treinar árvore com profundidade limitada (mais interpretável)
dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X_scaled, y)

# --- Visualizar a árvore ---
plot_tree(
    dt,
    feature_names=[f.replace('_', ' ') for f in wine_data.feature_names],
    class_names=['Cultivar A', 'Cultivar B', 'Cultivar C'],
    filled=True,          # Colorir por classe predominante
    rounded=True,         # Bordas arredondadas
    fontsize=8,
    ax=ax_tree,
    impurity=False,       # Ocultar Gini para simplificar
    precision=2
)
ax_tree.set_title('Árvore de Decisão (max_depth=3)\nConhecimento Descoberto pela Mineração',
                  fontsize=12, fontweight='bold')

# --- Importância das Features ---
feature_imp = pd.Series(
    dt.feature_importances_,
    index=wine_data.feature_names
).sort_values(ascending=True)  # Ordenar para gráfico horizontal

# Colorir barras por importância
colors_imp = plt.cm.RdYlGn(
    [v / feature_imp.max() for v in feature_imp]
)
bars = ax_imp.barh(feature_imp.index, feature_imp.values,
                   color=colors_imp, edgecolor='gray', linewidth=0.5)
# Adicionar valores nas barras
for bar, val in zip(bars, feature_imp.values):
    if val > 0.01:
        ax_imp.text(val + 0.005, bar.get_y() + bar.get_height()/2.,
                    f'{val:.3f}', va='center', fontsize=8)
ax_imp.set_title('Importância das Features (Gini Impurity)\nFactor mais relevantes para classificação',
                 fontsize=12, fontweight='bold')
ax_imp.set_xlabel('Importância Relativa')
ax_imp.axvline(x=feature_imp.mean(), color='red', linestyle='--',
               alpha=0.7, label=f'Média ({feature_imp.mean():.3f})')
ax_imp.legend()

plt.tight_layout()
plt.show()

# Resumo das features mais importantes
print('\nTop 5 Features mais Importantes:')
top5 = feature_imp.sort_values(ascending=False).head(5)
for feat, imp in top5.items():
    print(f'  {feat:40s}: {imp:.4f} ({imp*100:.1f}%)')

In [ ]:
# ============================================================
# FRONTEIRAS DE DECISÃO — PLOTLY INTERATIVO
# ============================================================

# Selecionar as 2 features mais importantes para visualização 2D
top2_features = feature_imp.sort_values(ascending=False).index[:2].tolist()
feat_idx = [list(wine_data.feature_names).index(f) for f in top2_features]

X_2d = X_scaled[:, feat_idx]  # Subconjunto 2D

# Treinar classificador nas 2 features selecionadas
clf_2d = DecisionTreeClassifier(max_depth=4, random_state=42)
clf_2d.fit(X_2d, y)

# Criar grade de pontos para visualizar fronteiras
h = 0.05  # Resolução da grade
x_min, x_max = X_2d[:, 0].min() - 0.5, X_2d[:, 0].max() + 0.5
y_min, y_max = X_2d[:, 1].min() - 0.5, X_2d[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                      np.arange(y_min, y_max, h))

# Predizer para cada ponto da grade
Z = clf_2d.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Plotar com Plotly (interativo)
fig_plotly = go.Figure()

# Camada de fundo (fronteiras de decisão)
fig_plotly.add_trace(go.Heatmap(
    x=np.arange(x_min, x_max, h),
    y=np.arange(y_min, y_max, h),
    z=Z,
    colorscale=[[0, 'lightblue'], [0.5, 'lightyellow'], [1, 'lightgreen']],
    opacity=0.5,
    showscale=False,
    name='Fronteiras'
))

# Pontos de dados por classe
colors_classes = ['#1565C0', '#E65100', '#2E7D32']
class_names_plot = ['Cultivar A', 'Cultivar B', 'Cultivar C']
for class_id in range(3):
    mask_c = y == class_id
    fig_plotly.add_trace(go.Scatter(
        x=X_2d[mask_c, 0],
        y=X_2d[mask_c, 1],
        mode='markers',
        name=class_names_plot[class_id],
        marker=dict(color=colors_classes[class_id], size=8,
                    line=dict(color='white', width=1))
    ))

fig_plotly.update_layout(
    title=f'Fronteiras de Decisão — {top2_features[0]} vs. {top2_features[1]}',
    xaxis_title=f'{top2_features[0]} (normalizado)',
    yaxis_title=f'{top2_features[1]} (normalizado)',
    width=800, height=550,
    legend=dict(x=0.01, y=0.99)
)
fig_plotly.show()
print(f'Features utilizadas: {top2_features[0]} e {top2_features[1]}')

## 📈 7. Análise de Complexidade e Escalabilidade <a id='complexidade'></a>

Analisamos como o tempo de treinamento de diferentes algoritmos escala
com o tamanho do dataset — informação crucial para aplicações em larga escala.


In [ ]:
# ============================================================
# ANÁLISE DE ESCALABILIDADE — TEMPO vs. TAMANHO DO DATASET
# ============================================================

# Tamanhos de dataset a serem testados
dataset_sizes = [100, 500, 1000, 5000, 10000, 50000]

# Algoritmos selecionados para a análise de escalabilidade
scale_clfs = {
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1),
    'Logistic Reg.': LogisticRegression(max_iter=500, random_state=42),
    'KNN (k=5)':     KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes':   GaussianNB()
}

# Dicionário para armazenar tempos
timing_results = {name: [] for name in scale_clfs}

print('Medindo tempo de treinamento por tamanho de dataset...')
print(f"{'Tamanho':>10} |", ' | '.join(f'{n[:8]:>10}' for n in scale_clfs))
print('-' * (12 + 13 * len(scale_clfs)))

for size in dataset_sizes:
    # Gerar dataset sintético com make_classification
    X_syn, y_syn = make_classification(
        n_samples=size,
        n_features=20,
        n_informative=10,
        n_classes=3,
        n_clusters_per_class=1,
        random_state=42
    )
    # Normalizar
    X_syn_s = StandardScaler().fit_transform(X_syn)

    row_times = []
    for name, clf in scale_clfs.items():
        # Medir tempo de fit
        t0 = time.time()
        clf.fit(X_syn_s, y_syn)
        elapsed = time.time() - t0
        timing_results[name].append(elapsed)
        row_times.append(elapsed)

    print(f'{size:>10} |', ' | '.join(f'{t:>10.4f}' for t in row_times))

# --- Plotar curvas de escalabilidade ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

colors_scale = plt.cm.Set1(np.linspace(0, 1, len(scale_clfs)))

for (name, times_list), color in zip(timing_results.items(), colors_scale):
    # Escala linear
    ax1.plot(dataset_sizes, times_list, 'o-', label=name, color=color,
             linewidth=2, markersize=7)
    # Escala log-log
    ax2.loglog(dataset_sizes, times_list, 'o-', label=name, color=color,
               linewidth=2, markersize=7)

ax1.set_title('Tempo de Treinamento vs. Tamanho do Dataset\n(Escala Linear)', fontweight='bold')
ax1.set_xlabel('Número de Amostras')
ax1.set_ylabel('Tempo de Treinamento (segundos)')
ax1.legend(fontsize=9)
ax1.xaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

ax2.set_title('Tempo de Treinamento vs. Tamanho do Dataset\n(Escala Log-Log)', fontweight='bold')
ax2.set_xlabel('Número de Amostras (log)')
ax2.set_ylabel('Tempo de Treinamento (log s)')
ax2.legend(fontsize=9)
ax2.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()
print('Análise de escalabilidade concluída!')

## 🎯 8. Exercícios Práticos <a id='exercicios'></a>

Complete os exercícios abaixo. Eles reforçam os conceitos vistos neste módulo.


In [ ]:
# ============================================================
# EXERCÍCIOS PRÁTICOS — TEMPLATES PARA COMPLETAR
# ============================================================

from sklearn.datasets import load_breast_cancer, load_digits

# ---------------------------------------------------------
# EXERCÍCIO 1: Carregar outro dataset e realizar EDA
# ---------------------------------------------------------
print('EXERCÍCIO 1: EDA no Dataset Breast Cancer')
print('Objetivo: Repetir o processo de EDA em outro dataset.')
print()

# TODO: Carregue o dataset breast_cancer com load_breast_cancer()
# TODO: Converta para DataFrame e mostre shape, head(), dtypes
# TODO: Verifique valores ausentes
# TODO: Mostre a distribuição das classes
# TODO: Crie pelo menos 2 visualizações

# Exemplo de início:
cancer = load_breast_cancer()
df_cancer = pd.DataFrame(cancer.data, columns=cancer.feature_names)
df_cancer['target'] = cancer.target

print(f'Shape: {df_cancer.shape}')
print(f'Classes: {cancer.target_names}')
print('\nSUA VEZ: Complete a EDA abaixo!')
# ... seu código aqui ...

print()
print('=' * 50)

# ---------------------------------------------------------
# EXERCÍCIO 2: Adicionar novo classificador
# ---------------------------------------------------------
print('EXERCÍCIO 2: Adicionar MLP ao Comparativo de Classificadores')
print('Objetivo: Incluir MLPClassifier na comparação do Módulo.')
print()

# TODO: Importe MLPClassifier de sklearn.neural_network
# TODO: Adicione ao dicionário de classificadores
# TODO: Execute cross-validation e compare com os demais
# TODO: Atualize os gráficos de comparação

print('Dica: from sklearn.neural_network import MLPClassifier')
print('Dica: MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500)')
print('\nSUA VEZ: Implemente o exercício 2 abaixo!')
# ... seu código aqui ...

print()
print('=' * 50)

# ---------------------------------------------------------
# EXERCÍCIO 3: K-Fold CV Manual (sem sklearn)
# ---------------------------------------------------------
print('EXERCÍCIO 3: Implementar K-Fold CV Manualmente')
print('Objetivo: Entender como funciona a validação cruzada por dentro.')
print()
print("""def kfold_cv_manual(X, y, clf, k=5):
    # TODO: Divida os índices em k partes iguais (np.array_split)
    # TODO: Para cada fold:
    #   - Use o fold como teste
    #   - Use os demais folds como treino
    #   - Treine o classificador e calcule a acurácia
    # TODO: Retorne a lista de acurácias e a média
    scores = []
    indices = np.arange(len(X))
    folds = np.array_split(indices, k)
    for i in range(k):
        test_idx = folds[i]
        train_idx = np.concatenate([folds[j] for j in range(k) if j != i])
        # SEU CÓDIGO AQUI
        pass
    return scores, np.mean(scores)
""")
print('\nSUA VEZ: Implemente a função e teste com o dataset Wine!')

## 📚 Resumo e Próximos Passos

### 🎯 Principais Aprendizados deste Notebook

| Conceito | O que aprendemos |
|----------|------------------|
| **EDA** | Visualizar distribuições, correlações e separabilidade de classes |
| **CRISP-DM** | Estruturar projetos de DM em 6 fases claras |
| **Comparação de Modelos** | Avaliar múltiplos algoritmos com CV e múltiplas métricas |
| **Interpretabilidade** | Árvores de decisão e importância de features |
| **Escalabilidade** | O tempo de treino cresce de forma diferente por algoritmo |

### 🔗 Próximo Módulo

**Módulo 2: Pré-processamento de Dados**:
- Tratamento de valores ausentes (MCAR, MAR, MNAR)
- Detecção e tratamento de outliers
- Normalização, escalonamento e transformações
- Seleção de atributos e redução de dimensionalidade
- Balanceamento de classes com SMOTE

### 📖 Referências

- FAYYAD et al. (1996). From Data Mining to Knowledge Discovery in Databases. *AI Magazine*, 17(3).
- WIRTH & HIPP (2000). CRISP-DM: Towards a Standard Process Model for Data Mining.
- HAN, PEI & KAMBER (2011). *Data Mining: Concepts and Techniques*. 3. ed. Morgan Kaufmann.
- CHEN & GUESTRIN (2016). XGBoost: A Scalable Tree Boosting System. *KDD'16*.
- LUNDBERG & LEE (2017). A Unified Approach to Interpreting Model Predictions. *NeurIPS*.
- Documentação scikit-learn: https://scikit-learn.org/stable/

---
